## **Install & Import Required Libraries**

In [1]:
# Install libraries (XGBoost & Scikit-Learn)
!pip install xgboost scikit-learn pandas numpy joblib

import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

Libraries imported successfully!


# Load the PaySim **Dataset**

In [8]:
import pandas as pd

# Load dataset directly from local session storage
df = pd.read_csv('dataset.csv')
print("Dataset Shape:", df.shape)
df.head()

Dataset Shape: (3479442, 11)


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0.0,0.0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0.0,0.0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1.0,0.0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1.0,0.0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0.0,0.0


# **Data Cleaning & Feature Engineering**

In [12]:
# 1. Filter TRANSFER & CASH_OUT (100% of PaySim fraud happens here)
df_filtered = df[df['type'].isin(['TRANSFER', 'CASH_OUT'])].copy()

# 2. Engineer Balance Error Features
df_filtered['errorBalanceOrig'] = df_filtered['newbalanceOrig'] + df_filtered['amount'] - df_filtered['oldbalanceOrg']
df_filtered['errorBalanceDest'] = df_filtered['oldbalanceDest'] + df_filtered['amount'] - df_filtered['newbalanceDest']

# 3. One-Hot Encode 'type' (TRANSFER = 1, CASH_OUT = 0)
df_filtered = pd.get_dummies(df_filtered, columns=['type'], drop_first=True)

# 4. Drop string identifiers & flagged target
cols_to_drop = ['nameOrig', 'nameDest', 'isFlaggedFraud']
df_filtered = df_filtered.drop(columns=cols_to_drop, errors='ignore')

# Convert booleans to 1/0
df_filtered = df_filtered.astype(float)

print("Cleaned Dataset Shape:", df_filtered.shape)
df_filtered.head()

Cleaned Dataset Shape: (1527257, 10)


,step,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,errorBalanceOrig,errorBalanceDest,type_TRANSFER
2,1.0,181.00,181.0,0.0,0.0,0.00,1.0,0.00,181.0,1.0
3,1.0,181.00,181.0,0.0,21182.0,0.00,1.0,0.00,21363.0,0.0
15,1.0,229133.94,15325.0,0.0,5083.0,51513.44,0.0,213808.94,182703.5,0.0
19,1.0,215310.30,705.0,0.0,22425.0,0.00,0.0,214605.30,237735.3,1.0
24,1.0,311685.89,10835.0,0.0,6267.0,2719172.89,0.0,300850.89,-2401220.0,1.0


# **Save Clean Dataset for File 2**

In [13]:
# Save cleaned features locally in Colab
df_filtered.to_csv('clean_dataset.csv', index=False)

# Download clean_dataset.csv to your computer
files.download('clean_dataset.csv')
print("clean_dataset.csv exported successfully!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

clean_dataset.csv exported successfully!


# **Stratified Train/Test Split & Feature Scaling**

In [16]:
# 1. Clean missing values from target y and features X
df_clean = df_filtered.dropna(subset=['isFraud'])

# 2. Separate Features (X) and Target (y)
X = df_clean.drop(columns=['isFraud'])
y = df_clean['isFraud'].astype(int)

# 3. Stratified Train-Test Split (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# 4. Scale continuous numerical features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("X_train shape:", X_train_scaled.shape)
print("X_test shape:", X_test_scaled.shape)

X_train shape: (1221804, 9)
X_test shape: (305452, 9)


# **Benchmark Multiple ML Algorithms**

In [17]:
# Define baseline algorithms
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
}

results = []

for name, model in models.items():
    print(f"Training {name}...")

    # Logistic Regression requires scaled inputs; Tree models can use raw X_train
    if name == "Logistic Regression":
        model.fit(X_train_scaled, y_train)
        preds = model.predict(X_test_scaled)
        probs = model.predict_proba(X_test_scaled)[:, 1]
    else:
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        probs = model.predict_proba(X_test)[:, 1]

    # Calculate evaluation metrics
    auc = roc_auc_score(y_test, probs)
    report = classification_report(y_test, preds, output_dict=True)

    results.append({
        "Model": name,
        "Precision (Fraud)": round(report['1']['precision'], 4),
        "Recall (Fraud)": round(report['1']['recall'], 4),
        "F1-Score (Fraud)": round(report['1']['f1-score'], 4),
        "ROC-AUC": round(auc, 4)
    })

# Render performance comparison table
benchmark_df = pd.DataFrame(results)
print("\n--- BENCHMARK RESULTS ---")
print(benchmark_df.to_string(index=False))

Training Logistic Regression...
Training Random Forest...
Training XGBoost...

--- BENCHMARK RESULTS ---
              Model  Precision (Fraud)  Recall (Fraud)  F1-Score (Fraud)  ROC-AUC
Logistic Regression             0.8986          0.4283            0.5801   0.9749
      Random Forest             0.9983          0.9896            0.9939   0.9948
            XGBoost             0.9845          0.9862            0.9853   0.9950
